# Lecture 2: Image Classification

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 2, with material from Stanford CS231n 2026, Lecture 2.*

**Image classification:** given an image, choose one label from a fixed set (cat, dog, truck, …). It's the core task of computer vision, and a building block for almost everything else.

**What you'll learn**
1. Why image classification is hard.
2. The **data-driven approach**: learn from examples instead of writing rules.
3. The **nearest neighbor** and **k-nearest neighbor** classifiers.
4. How to choose **hyperparameters** properly (validation sets, cross-validation).
5. Why nearest neighbor on raw pixels doesn't work well for real images.

We'll use a small dataset of **handwritten digits** (8×8 pixel images) so everything runs in seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from sklearn.datasets import load_digits

## 1. The data

1,797 images of handwritten digits, each 8×8 pixels with values from 0 (white) to 16 (black).

In [ ]:
digits = load_digits()
images = digits.images          # shape (1797, 8, 8)
labels = digits.target          # the correct digit for each image

print("number of images:", len(images))
print("one image is a", images[0].shape, "grid of numbers")

fig, axes = plt.subplots(1, 10, figsize=(10, 1.5))
for digit in range(10):
    first_example = np.where(labels == digit)[0][0]
    axes[digit].imshow(images[first_example], cmap="gray_r")
    axes[digit].set_title(digit)
    axes[digit].axis("off")
plt.show()

Most methods in this lecture treat an image as **one long list of numbers**, so we **flatten** each 8×8 image into 64 numbers. Then we split the data into a **training set** (to learn from) and a **test set** (to check how well we do on images we've never seen).

In [ ]:
X = images.reshape(len(images), 64)       # each row is one flattened image
y = labels

rng = np.random.default_rng(0)
order = rng.permutation(len(X))           # shuffle before splitting
train_idx, test_idx = order[:1300], order[1300:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("training images:", X_train.shape, "  test images:", X_test.shape)

## 2. Why is image classification hard?

We see a digit; the computer sees 64 numbers. That's the **semantic gap**. Worse, the numbers change a lot under changes that don't change the label at all:
* **viewpoint** (the camera moves: every pixel changes)
* **illumination** (lighting)
* **background clutter**, **occlusion** (part of the object is hidden)
* **deformation** (cats are flexible!), **intraclass variation** (many kinds of chairs)

Let's measure it. The **distance** between two images: subtract them pixel by pixel, square, add up, take the square root (the **L2 distance**).

In [ ]:
def l2_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

zero = images[0]                                              # a '0'
zero_shifted = ndimage.shift(zero, (0, 1), order=0)          # the same '0', moved 1 pixel to the right

print("distance from the '0' to itself shifted 1 pixel:", round(l2_distance(zero, zero_shifted), 1))

# How close is the nearest image of a DIFFERENT digit?
other_digits = images[labels != 0]
closest = min(l2_distance(zero, other) for other in other_digits)
print("distance from the '0' to the closest NON-zero image:", round(closest, 1))

**What we see:** moving the digit by just one pixel changes it *more* than swapping it for a completely different digit. Pixel values don't directly reflect what's in the picture.

## 3. The data-driven approach

We can't write rules like "a 0 is a closed loop" that work for every handwriting style. Instead:
1. **Collect** a dataset of images with labels.
2. **Train** a classifier with machine learning.
3. **Evaluate** it on new images.

Every classifier then has two functions:
```python
def train(images, labels):        # learn from examples
    ...
    return model

def predict(model, test_images):  # use what was learned
    ...
    return predicted_labels
```

## 4. Our first classifier: nearest neighbor

* **Train:** memorize all the training images and labels.
* **Predict:** for a new image, find the **most similar** training image and copy its label.

"Most similar" needs a distance. The lecture uses the **L1 (Manhattan) distance**: compare pixel by pixel and add up the absolute differences:

$$d_1(I_1, I_2) = \sum_{p}|I_1^p - I_2^p|$$

Here's the lecture's 4×4 example:

In [ ]:
test_image = np.array([[56, 32, 10, 18],
                       [90, 23, 128, 133],
                       [24, 26, 178, 200],
                       [2, 0, 255, 220]])
training_image = np.array([[10, 20, 24, 17],
                           [8, 10, 89, 100],
                           [12, 16, 178, 170],
                           [4, 32, 233, 112]])

differences = np.abs(test_image - training_image)
print("pixel-wise absolute differences:\n", differences)
print("L1 distance =", differences.sum())

Now the classifier. We write it as plainly as possible: for each test image, compute its distance to every training image, and pick the closest.

In [ ]:
class NearestNeighbor:
    def train(self, X, y):
        # "Training" is just remembering the data.
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = []
        for test_image in X:
            distances = np.sum(np.abs(self.X_train - test_image), axis=1)   # L1 distance to every training image
            closest = np.argmin(distances)                                   # index of the smallest distance
            predictions.append(self.y_train[closest])
        return np.array(predictions)

model = NearestNeighbor()
model.train(X_train, y_train)
predictions = model.predict(X_test)
print(f"test accuracy: {np.mean(predictions == y_test):.1%}")

About 98% with no knowledge about digits at all: the data did all the work. Let's look at some test images next to their nearest training image:

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i in range(8):
    test_image = X_test[i]
    distances = np.sum(np.abs(X_train - test_image), axis=1)
    closest = np.argmin(distances)
    axes[0, i].imshow(test_image.reshape(8, 8), cmap="gray_r")
    axes[0, i].set_title(f"test: {y_test[i]}", fontsize=9)
    axes[1, i].imshow(X_train[closest].reshape(8, 8), cmap="gray_r")
    axes[1, i].set_title(f"nearest: {y_train[closest]}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.show()

**How fast is it?** With $N$ training images:
* **training** takes almost no time (just store the data);
* **prediction** compares against all $N$ images, so it gets slower as the dataset grows.

That's **backwards**: we train once, but predict all the time (often on a phone). We want classifiers that are fast at prediction, even if training is slow. Neural networks are like that.

## 5. k-nearest neighbors

A single nearest neighbor can be misleading: if it's mislabeled or unusual, we copy its mistake. **k-nearest neighbors (kNN):** find the $k$ closest training images and take a **majority vote**.

Let's see the difference in 2-D, where we can draw it. Three clusters of points, with a few deliberately mislabeled points ("outliers"):

In [ ]:
rng = np.random.default_rng(3)
centers = [[0, 0], [3, 0], [1.5, 2.5]]
points = np.vstack([rng.normal(center, 0.7, size=(30, 2)) for center in centers])
point_labels = np.repeat([0, 1, 2], 30)
point_labels[[5, 40, 70]] = [1, 2, 0]                # three mislabeled outliers

def knn_predict(train_points, train_labels, query, k):
    distances = np.sqrt(np.sum((train_points - query) ** 2, axis=1))
    nearest = np.argsort(distances)[:k]              # indices of the k closest points
    votes = np.bincount(train_labels[nearest], minlength=3)
    return np.argmax(votes)                          # the label with the most votes

print("prediction for the point (0, 0) with k = 5:", knn_predict(points, point_labels, np.array([0, 0]), 5))

To see the **decision regions**, classify every point on a grid and color it:

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-2, 5, 120), np.linspace(-2, 4.5, 120))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, k in zip(axes, [1, 5]):
    regions = np.array([knn_predict(points, point_labels, q, k) for q in grid])
    ax.contourf(grid_x, grid_y, regions.reshape(grid_x.shape), alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5], cmap="brg")
    ax.scatter(points[:, 0], points[:, 1], c=point_labels, cmap="brg", edgecolors="k", s=25)
    ax.set_title(f"k = {k}")
plt.show()

**What we see:** with $k = 1$, each outlier carves out its own little island of the wrong color. With $k = 5$, the islands disappear and the boundaries are smoother.

## 6. Choosing the distance: L1 vs. L2

$$\text{L1:}\ d_1 = \sum_p|I_1^p - I_2^p|\qquad\qquad\text{L2:}\ d_2 = \sqrt{\sum_p(I_1^p - I_2^p)^2}$$

* **L2** is the straight-line distance ("as the crow flies").
* **L1** is the distance walking along a grid of city streets.

The two often give different answers. Here's the lecture's example: from the origin, the points $(1, 0)$ and $(0.5, 0.5)$ are equally far in L1, but not in L2.

In [ ]:
def l1_distance(a, b):
    return np.sum(np.abs(a - b))

origin, A, B = np.array([0, 0]), np.array([1, 0]), np.array([0.5, 0.5])
print("L1: origin to A =", l1_distance(origin, A), ",  origin to B =", l1_distance(origin, B))
print("L2: origin to A =", l2_distance(origin, A), ",  origin to B =", round(l2_distance(origin, B), 3))

The **distance metric** and the value of **k** are choices we have to make. Which ones work best for our digits?

## 7. Hyperparameters

**Hyperparameters** are choices about the algorithm (like $k$ and the distance) that we set **before** training rather than learn from the data. How should we choose them? The lecture discusses four ideas:

* **Idea 1: pick what works best on the training data.** ❌ Bad: $k = 1$ is *always* perfect on training data, since every image's nearest neighbor is itself.
* **Idea 2: pick what works best on the test data.** ❌ Bad: then the test score is no longer an honest estimate of performance on new data.
* **Idea 3: split off a validation set**; choose on validation, report on test. ✅
* **Idea 4: cross-validation:** rotate which part of the data is the validation set, and average. ✅ (Good for small datasets.)

Let's check Idea 1's problem directly:

In [ ]:
def knn_classify(X_train, y_train, X_query, k, distance="L1"):
    # k-nearest-neighbor predictions for every row of X_query.
    predictions = []
    for query in X_query:
        if distance == "L1":
            distances = np.sum(np.abs(X_train - query), axis=1)
        else:
            distances = np.sqrt(np.sum((X_train - query) ** 2, axis=1))
        nearest = np.argsort(distances)[:k]
        votes = np.bincount(y_train[nearest], minlength=10)
        predictions.append(np.argmax(votes))
    return np.array(predictions)

train_accuracy = np.mean(knn_classify(X_train, y_train, X_train[:300], k=1) == y_train[:300])
print(f"k = 1, accuracy on (part of) the TRAINING data: {train_accuracy:.0%}  <- always perfect, so it tells us nothing")

**Idea 3: a validation set.** We split our training data again: 1,000 images to train on, 300 to validate on. The test set stays locked away.

In [ ]:
X_tr, y_tr = X_train[:1000], y_train[:1000]
X_val, y_val = X_train[1000:], y_train[1000:]

print(f"{'k':>3s}  {'L1 validation':>14s}  {'L2 validation':>14s}")
for k in [1, 3, 5, 7, 11, 21]:
    acc_l1 = np.mean(knn_classify(X_tr, y_tr, X_val, k, "L1") == y_val)
    acc_l2 = np.mean(knn_classify(X_tr, y_tr, X_val, k, "L2") == y_val)
    print(f"{k:3d}  {acc_l1:14.1%}  {acc_l2:14.1%}")

**Idea 4: 5-fold cross-validation.** Split the training data into 5 parts ("folds"). Each fold takes a turn as the validation set while we train on the other four. We get 5 accuracies per setting, which also shows how much the result depends on the split.

In [ ]:
folds_X = np.array_split(X_train, 5)
folds_y = np.array_split(y_train, 5)
k_values = [1, 3, 5, 7, 11, 21]
results = {}

for k in k_values:
    accuracies = []
    for i in range(5):
        X_val_fold, y_val_fold = folds_X[i], folds_y[i]
        X_tr_fold = np.vstack([folds_X[j] for j in range(5) if j != i])        # the other 4 folds
        y_tr_fold = np.concatenate([folds_y[j] for j in range(5) if j != i])
        predictions = knn_classify(X_tr_fold, y_tr_fold, X_val_fold, k, "L2")
        accuracies.append(np.mean(predictions == y_val_fold))
    results[k] = accuracies
    print(f"k = {k:2d}: mean {np.mean(accuracies):.1%}  (folds: {', '.join(f'{a:.0%}' for a in accuracies)})")

In [ ]:
means = [np.mean(results[k]) for k in k_values]
stds = [np.std(results[k]) for k in k_values]
plt.figure(figsize=(6, 3.5))
for k in k_values:
    plt.scatter([k] * 5, results[k], color="gray", s=10)       # each fold's result
plt.errorbar(k_values, means, yerr=stds, marker="o", capsize=4)  # mean +/- standard deviation
plt.xlabel("k")
plt.ylabel("validation accuracy")
plt.title("5-fold cross-validation (L2 distance)")
plt.show()

Now we choose the best $k$ **and only then** use the test set, **once**:

In [ ]:
best_k = k_values[int(np.argmax(means))]
test_accuracy = np.mean(knn_classify(X_train, y_train, X_test, best_k, "L2") == y_test)
print(f"chosen k = {best_k};  test accuracy (used once, at the very end): {test_accuracy:.1%}")

**Notice** that the differences between nearby $k$ values are about as big as the fold-to-fold spread, so don't read too much into small differences. On these clean digits a small $k$ works well; on harder data like CIFAR-10, CS231n finds $k \approx 7$ is best.

## 8. Why kNN on raw pixels isn't used for real images

**Reason 1: it's slow at prediction time** (Section 4).

**Reason 2: pixel distances don't match meaning.** We already saw that a 1-pixel shift moves a digit a long way. Let's see what it does to accuracy: shift every test digit one pixel to the right.

In [ ]:
X_test_shifted = np.array([ndimage.shift(img.reshape(8, 8), (0, 1), order=0).ravel() for img in X_test])

normal_accuracy = np.mean(knn_classify(X_train, y_train, X_test, best_k, "L2") == y_test)
shifted_accuracy = np.mean(knn_classify(X_train, y_train, X_test_shifted, best_k, "L2") == y_test)
print(f"original test digits: {normal_accuracy:.1%}")
print(f"shifted by 1 pixel:   {shifted_accuracy:.1%}")

A change no human would even notice makes the classifier fail far more often.

**Reason 3: the curse of dimensionality.** For the training points to cover the space densely, the number of points needed grows **exponentially** with the number of dimensions. With points spaced every 0.25 along each axis:

In [ ]:
for dimensions in [1, 2, 3, 10, 64]:
    print(f"{dimensions:2d} dimensions: {4 ** dimensions:.2e} points needed")

Our 8×8 digits have 64 dimensions; a small 32×32 color image has 3,072. No dataset could cover that space.

**But kNN works well on good features.** If we compare images using features from a trained neural network (instead of raw pixels), nearest neighbors are very useful. They're used today for image search, finding duplicates, and evaluating self-supervised models.

## 9. kNN on CIFAR-10 (optional, run in Colab)

CIFAR-10 has 32×32 color images of 10 classes (airplane, car, bird, cat, …). kNN on raw pixels gets roughly 35–40% accuracy (10% would be random guessing), far below a convolutional network (over 90%).

In [ ]:
# [PyTorch] Needs: pip install torch torchvision (downloads CIFAR-10, about 170 MB).
import torch, torchvision
train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

X_train_c = torch.tensor(train_set.data[:10000], dtype=torch.float32).reshape(10000, -1)   # 10,000 training images
y_train_c = torch.tensor(train_set.targets[:10000])
X_test_c = torch.tensor(test_set.data[:500], dtype=torch.float32).reshape(500, -1)
y_test_c = torch.tensor(test_set.targets[:500])

distances = torch.cdist(X_test_c, X_train_c)               # all test-to-train L2 distances at once
nearest = distances.topk(5, largest=False).indices          # the 5 closest training images
predictions = torch.mode(y_train_c[nearest], dim=1).values  # majority vote
print(f"CIFAR-10, 5-NN on raw pixels: {(predictions == y_test_c).float().mean():.1%}")

## Summary
* **Image classification:** image in, one label out. Hard because of the semantic gap and all the ways images vary.
* **Data-driven approach:** learn from labeled examples with `train()` and `predict()`.
* **Nearest neighbor:** memorize, then copy the label of the closest training image. Instant to train, slow to predict.
* **kNN:** a majority vote of the $k$ closest; smoother boundaries, robust to outliers.
* **Hyperparameters** ($k$, the distance) are chosen on a **validation set** or with **cross-validation**. **Use the test set only once, at the end.**
* **Raw pixel distances are a poor measure of similarity**, which is why we need learned features.

### Check your understanding
1. Why is $k = 1$ always 100% accurate on the training data?
2. What's wrong with choosing $k$ by looking at test accuracy?
3. Why is nearest neighbor's speed "backwards"?
4. In the 2-D picture, why does $k = 5$ remove the small islands?

<details><summary>Answers</summary>

1. Every training image is its own nearest neighbor (distance 0), so it always gets its own label.
2. You've used the test set to make a choice, so its score is no longer an honest estimate of performance on new data; it will be optimistic.
3. Training is instant but prediction compares with every training image, and we predict far more often than we train.
4. A single mislabeled point is outvoted by its correctly labeled neighbors.
</details>

**Next:** Lecture 3, linear classifiers.